# Notebook 4 — EDA with SQL

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')
df = pd.read_csv('spacex_launches_raw.csv')
df.to_sql('launches', conn, index=False, if_exists='replace')
print(f'{len(df)} rows loaded into SQLite table `launches`')

98 rows loaded into SQLite table `launches`


### Query 1 — Unique launch sites

In [2]:
print(pd.read_sql_query('SELECT DISTINCT Launch_Site FROM launches', conn))

   Launch_Site
0  CCAFS LC-40
1  VAFB SLC-4E
2   KSC LC-39A
3   Boca Chica


### Query 2 — Total payload mass delivered

In [3]:
print(pd.read_sql_query('SELECT SUM(Payload_Mass_kg) AS total_payload_kg FROM launches', conn))

   total_payload_kg
0            376406


### Query 3 — Average payload mass by launch site

In [4]:
print(pd.read_sql_query('SELECT Launch_Site, ROUND(AVG(Payload_Mass_kg)) AS avg_payload_kg FROM launches GROUP BY Launch_Site', conn))

   Launch_Site  avg_payload_kg
0   Boca Chica          4765.0
1  CCAFS LC-40          3748.0
2   KSC LC-39A          4178.0
3  VAFB SLC-4E          3403.0


### Query 4 — Landing success rate by site

In [5]:
print(pd.read_sql_query('''SELECT Launch_Site, COUNT(*) AS launches, SUM(Class) AS landed,
       ROUND(SUM(Class)*100.0/COUNT(*), 1) AS success_pct
       FROM launches GROUP BY Launch_Site ORDER BY success_pct DESC''', conn))

   Launch_Site  launches  landed  success_pct
0   Boca Chica         9       9        100.0
1   KSC LC-39A        16      13         81.3
2  VAFB SLC-4E        20      16         80.0
3  CCAFS LC-40        53      36         67.9


### Query 5 — Booster reuse statistics

In [6]:
print(pd.read_sql_query('SELECT ROUND(AVG(Booster_Landings),1) AS avg_prior_landings, MAX(Booster_Landings) AS max_prior_landings FROM launches', conn))

   avg_prior_landings  max_prior_landings
0                 1.5                   5


### Query 6 — Failure rate by orbit type

In [7]:
print(pd.read_sql_query('''SELECT Orbit_Type, COUNT(*) AS launches,
       SUM(CASE WHEN Class=0 THEN 1 ELSE 0 END) AS failures,
       ROUND(SUM(CASE WHEN Class=0 THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS failure_pct
       FROM launches GROUP BY Orbit_Type ORDER BY failure_pct DESC''', conn))

  Orbit_Type  launches  failures  failure_pct
0        HEO         8         6         75.0
1        SSO        16         9         56.3
2        LEO        42         7         16.7
3        ISS        16         1          6.3
4        GEO        16         1          6.3
